# Can we predict `passed_quiz`?
Target: whether a student passes the quiz closing a tutoring session (one row = one session).

**Plan**
1. Check the data (repeated students, odd values).
2. Evaluate with **student-grouped** cross-validation, since the same student appears in many rows. Show how much a naive random split inflates the score.
3. Compare a base-rate baseline, logistic regression and gradient boosting.
4. Report what drives the prediction and what it means.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score
from sklearn.inspection import permutation_importance
SEED = 0
df = pd.read_csv('data.csv')
pd.read_csv('columns.csv')

## 1. Data checks

In [ ]:
print(df.shape, 'missing:', df.isna().sum().sum(), 'duplicate rows:', df.duplicated().sum())
g = df.groupby('student_key')
print('students:', df.student_key.nunique(), '| sessions/student (min, median, max):', tuple(g.size().agg(['min', 'median', 'max'])))
print('schools:', df.school_code.nunique(), '| pass rate:', df.passed_quiz.mean().round(3))
print('prior_score > 100:', (df.prior_score > 100).sum(), 'rows (kept)')
print('within-student std of prior_score (median):', g.prior_score.std().median().round(2), 'vs overall std', df.prior_score.std().round(2))
df[['prior_score', 'session_minutes', 'hints_used']].describe(percentiles=[.01, .99]).round(2)

- Students repeat (~7 sessions each). `prior_score` is described as a placement score but varies slightly per row (std ≈ 0.3 vs 11.6 overall), so it effectively fingerprints the student. A flexible model with a random row split can memorise students through it.
- Pass rates cluster by student, shown next.

In [ ]:
s = g.passed_quiz.agg(['mean', 'size'])
p0 = df.passed_quiz.mean()
print('variance of per-student pass rate:', s['mean'].var().round(3), '| expected if sessions were independent:', (p0 * (1 - p0) / s['size'].mean()).round(3))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
s['mean'].hist(bins=20, ax=ax[0]); ax[0].set_title('per-student pass rate')
df.groupby(pd.qcut(df.session_minutes, 10)).passed_quiz.mean().reset_index(drop=True).plot(marker='o', ax=ax[1], title='pass rate by session_minutes decile')
df.groupby('hints_used').passed_quiz.mean().plot(marker='o', ax=ax[2], title='pass rate by hints_used')
plt.tight_layout()

Students differ far more in pass rate than chance would give, so part of the signal is a stable student trait. A model must therefore be judged on **students it has not seen**. At session level, longer sessions and fewer hints go with passing.

## 2. Evaluation setup
Metrics: ROC AUC (ranking), log loss and Brier (calibration), accuracy vs the ~58% majority baseline. `student_key` is never a feature. Logistic regression uses the three numeric features; boosting uses all five (`school_code` as categorical).

In [ ]:
y = df.passed_quiz.values; groups = df.student_key.values
FEATS = ['enrol_year', 'school_code', 'prior_score', 'session_minutes', 'hints_used']
LR_FEATS = ['prior_score', 'session_minutes', 'hints_used']
X = df[FEATS]
def models():
    return {
        'base rate': (DummyClassifier(strategy='prior'), FEATS),
        'logistic': (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), LR_FEATS),
        'grad. boosting': (HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                           categorical_features=[FEATS.index('school_code')], random_state=SEED), FEATS),
    }
def evaluate(cv, use_groups):
    rows = {}
    for name, (m, cols) in models().items():
        p = cross_val_predict(m, X[cols], y, cv=cv, groups=groups if use_groups else None, method='predict_proba')[:, 1]
        rows[name] = dict(auc=roc_auc_score(y, p), logloss=log_loss(y, p), brier=brier_score_loss(y, p), acc=accuracy_score(y, p > .5))
    return pd.DataFrame(rows).T.round(3)
grp = evaluate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED), True)
rnd = evaluate(StratifiedKFold(5, shuffle=True, random_state=SEED), False)
print('Grouped by student (honest, new students)'); display(grp)
print('Random row split (leaky: same student in train and test)'); display(rnd)

The gap between the two tables for boosting is score that comes from recognising students, not from generalisable features. The grouped table is the one to trust.

In [ ]:
aucs = pd.concat([evaluate(StratifiedGroupKFold(5, shuffle=True, random_state=sd), True)['auc'] for sd in range(5)], axis=1)
pd.DataFrame({'mean AUC': aucs.mean(axis=1), 'sd across 5 CV seeds': aucs.std(axis=1)}).round(3)

## 3. What drives it?

In [ ]:
lr, _ = models()['logistic']; lr.fit(X[LR_FEATS], y)
print('logistic coefficients per 1 SD:'); display(pd.Series(lr[-1].coef_[0], index=LR_FEATS).round(3))
imp = []
for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, groups):
    m = models()['grad. boosting'][0].fit(X.iloc[tr], y[tr])
    imp.append(permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc', n_repeats=5, random_state=SEED).importances_mean)
pd.Series(np.mean(imp, axis=0), index=FEATS).sort_values().plot.barh(title='AUC drop when shuffled (held-out students)');

In [ ]:
m, cols = models()['logistic']
p = cross_val_predict(m, X[cols], y, cv=StratifiedGroupKFold(5, shuffle=True, random_state=SEED), groups=groups, method='predict_proba')[:, 1]
c = pd.DataFrame({'p': p, 'y': y}).groupby(pd.qcut(p, 10)).mean()
plt.plot(c.p, c.y, 'o-'); plt.plot([c.p.min(), c.p.max()], [c.p.min(), c.p.max()], '--k')
plt.xlabel('predicted'); plt.ylabel('observed'); plt.title('Calibration, logistic, held-out students');

## 4. Conclusion
Results on **unseen students** (5-fold student-grouped CV, repeated over 5 seeds; majority-class accuracy 0.579, base-rate AUC 0.5):

| model | AUC | log loss | accuracy |
|---|---|---|---|
| logistic (prior_score, session_minutes, hints_used) | 0.601 | 0.665 (base 0.681) | 0.598 |
| gradient boosting (all features) | 0.572 | 0.687 | 0.580 |

- **Yes, but only weakly.** The simple logistic model beats the base rate (AUC ~0.60, log loss slightly better, accuracy ~2 points above guessing). That is far from a reliable per-session prediction.
- The signal comes from `session_minutes` (longer sessions pass more, +0.30 per SD) and `hints_used` (more hints, fewer passes, -0.22 per SD). `prior_score` is nearly irrelevant (+0.06). `enrol_year` and `school_code` add nothing for new students.
- **Boosting only looked good under the wrong validation.** With a random row split its AUC is 0.653, but on unseen students it drops to 0.572, below the logistic model. It was recognising students (via the near-unique `prior_score`) rather than learning anything transferable. Any score reported from a random split of this data would be inflated.

**Caveats / next steps**
- `session_minutes` and `hints_used` are only complete once the session ends, so this is not an advance forecast; check they are known before the quiz if you want to use it live.
- Per-student pass rates vary much more than chance (variance 0.125 vs 0.035), so a student's *earlier* quiz results would probably predict much better than anything here. The data has no date or session order, so we could not test it; if timestamps exist, that is the next experiment.
- `prior_score` is documented as a placement score but varies slightly between a student's rows and exceeds 100 in 19 rows; worth asking how it is recorded.
- Associations are not causal (struggling students probably use more hints).
- The notebook is saved without cell outputs; the numbers above come from running these same cells.